# ProCIR training on Kaggle (T4)

Runs `procir_train.pipeline` (env check → train → export → eval) from the HCMUT_Fashion_MV repository.
**Kaggle is for pilots.** A T4 is sm75: bf16 is emulated and `causal-conv1d` does not build here, so steps are slow.
Run the proposal-scale experiments on the Docker server (`docker/run.sh`).

**Once, on your machine** (Kaggle CLI logged in):
```bash
python scripts/pack_kaggle.py code   && kaggle datasets create -p dist/kaggle/procir-code --dir-mode zip
python scripts/pack_kaggle.py images && kaggle datasets create -p data/images_official_hr --dir-mode zip
kaggle kernels push -p src/kaggle/train
```
After a code change: `python scripts/pack_kaggle.py code && kaggle datasets version -p dist/kaggle/procir-code --dir-mode zip -m "update"`.

**Each session** (Save Version → Save & Run All, GPU T4, Internet on) trains for up to `SESSION_HOURS`, saves
`runs/<RUN_NAME>/ckpt/latest.pt` to the output and stops. **To continue:** Add Input → *Your Work* → this notebook's
output, then Save Version again; the notebook copies the run back and resumes. When the last cell prints
`training finished`, set `STAGES = ["export", "eval"]` for the next session (eval on a T4 can take ~10 h per dataset).
Keep `RUN_NAME`, `CONFIG` and `OVERRIDES` identical across sessions of one run; the trainer refuses a resume otherwise.

In [ ]:
# ---- parameters ----
RUN_NAME = "kaggle_mt_align_seed42"
CONFIG = "configs/kaggle_t4.yaml"
OVERRIDES = []                 # extra config keys, e.g. ["cot=true", "seed=1"]
STAGES = ["train"]             # later session: ["export", "eval"]
EVAL_PROTOCOLS = ["upstream"]  # add "train" to also score at the training views/pixels
EVAL_BATCH_SIZE = 8
SESSION_HOURS = 11.0           # Kaggle ends GPU sessions at 12 h; keep a margin for setup and saving
INSTALL_FLA = True             # flash-linear-attention; removed again if it fails the self-test on this GPU
KEEP_STEP_CKPTS = False        # False: keep only ckpt/latest.pt in the output (smaller outputs)

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

T0 = time.time()
WORK, INPUT = Path("/kaggle/working"), Path("/kaggle/input")
ENV = dict(os.environ, TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")


def sh(cmd, check=True, cwd=None):
    """Run a command and stream its output into the notebook log."""
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(cmd if isinstance(cmd, str) else [str(c) for c in cmd], shell=isinstance(cmd, str),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENV, cwd=cwd)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
    return p.returncode


def find_under_input(rel, what, depth=4):
    """First /kaggle/input/<..up to depth..>/<rel>; the mount layout differs between Kaggle versions."""
    for d in range(depth + 1):
        hits = sorted(INPUT.glob("/".join(["*"] * d + [rel])))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{what}: no {rel} under {INPUT}; attach it with Add Input")


sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv")

In [ ]:
# ---- code (dataset procir-code) and images (dataset procir-images) ----
ROOT = WORK / "code"
shutil.rmtree(ROOT, ignore_errors=True)
shutil.copytree(find_under_input("src/procir_train/pipeline.py", "procir-code dataset").parents[2], ROOT)
IMAGE_ROOT = find_under_input("deepfashion", "procir-images dataset").parent
ENV["PYTHONPATH"] = f"{ROOT / 'src'}:{ROOT / 'external' / 'FashionMV'}"
print("code:", ROOT, "| images:", IMAGE_ROOT, sorted(p.name for p in IMAGE_ROOT.iterdir() if p.is_dir()))

In [ ]:
# ---- dependencies: kernels installed one by one so one failed build cannot block the others ----
sh(f'{sys.executable} -m pip install -q -U "transformers>=5.0" "peft>=0.13" "huggingface_hub<2.0" pyyaml')
if INSTALL_FLA:
    sh(f"{sys.executable} -m pip install -q flash-linear-attention", check=False)

In [ ]:
# ---- annotations (triplets + captions) from the FashionMV dataset on Hugging Face ----
from huggingface_hub import snapshot_download

snapshot_download("yuandaxia/FashionMV", repo_type="dataset", local_dir=str(WORK / "hf_data"),
                  allow_patterns=["*triplets.jsonl", "*captions.jsonl"])
DATA_DIR = sorted((WORK / "hf_data").rglob("train_triplets.jsonl"))[0].parent
print("annotations:", DATA_DIR, sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
# ---- self-test: one forward + backward of Qwen3.5 with LoRA on this GPU (also downloads the base model) ----
SELF_TEST = """
import torch
from procir_train.config import TrainConfig
from procir_train.encoder import load_encoder
from procir_train.prompts import caption_text, process_text
enc, proc = load_encoder(TrainConfig(), torch.device("cuda"))
x = [process_text(proc, caption_text(proc, "a red cotton t-shirt with a round neck and short sleeves"))] * 2
enc.encode_single(x).float().sum().backward()
torch.cuda.synchronize()
print("self-test OK")
"""
if sh([sys.executable, "-c", SELF_TEST], check=False) != 0:
    if not INSTALL_FLA:
        raise RuntimeError("self-test failed without flash-linear-attention; see the log above")
    print("self-test failed with flash-linear-attention; uninstalling it and using the PyTorch fallback")
    sh(f"{sys.executable} -m pip uninstall -y -q flash-linear-attention fla-core", check=False)
    sh([sys.executable, "-c", SELF_TEST])

In [ ]:
# ---- resume: copy this run back from a previous version's output, if one is attached ----
RUN_DIR = WORK / "runs" / RUN_NAME
if not (RUN_DIR / "ckpt" / "latest.pt").exists():
    try:
        prev = find_under_input(f"runs/{RUN_NAME}/ckpt/latest.pt", "previous output").parents[1]
    except FileNotFoundError:
        print("no previous output attached: starting a new run")
    else:
        shutil.copytree(prev, RUN_DIR, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("step_*.pt", "export", "eval_upstream"))
        print("restored", prev, "->", RUN_DIR)
        print((RUN_DIR / "STATUS").read_text() if (RUN_DIR / "STATUS").exists() else "")

In [ ]:
# ---- run the pipeline ----
budget = max(0.25, round(SESSION_HOURS - (time.time() - T0) / 3600, 2))
print(f"training budget for this session: {budget} h")
sets = [f"data_dir={DATA_DIR}", f"image_root={IMAGE_ROOT}", f"output_dir={RUN_DIR}",
        f"stop_after_hours={budget}", *OVERRIDES]
sh([sys.executable, "-m", "procir_train.pipeline", "--config", ROOT / CONFIG, "--set", *sets,
    "--stages", *STAGES, "--eval-protocols", *EVAL_PROTOCOLS, "--eval-batch-size", EVAL_BATCH_SIZE,
    "--allow-slow"], cwd=ROOT)

In [ ]:
# ---- summary; trim the output ----
import json

print((RUN_DIR / "STATUS").read_text() if (RUN_DIR / "STATUS").exists() else "no STATUS file")
log = RUN_DIR / "train_log.jsonl"
if log.exists():
    rows = [json.loads(l) for l in log.read_text().splitlines() if l.strip()]
    for r in [r for r in rows if "loss" in r][-5:]:
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
for f in sorted(RUN_DIR.rglob("*.json")):
    if f.parent.name.startswith("eval") or f.name.startswith("eval_"):
        print(f, f.read_text()[:2000])
if not KEEP_STEP_CKPTS:
    for f in (RUN_DIR / "ckpt").glob("step_*.pt"):
        f.unlink()
shutil.rmtree(ROOT, ignore_errors=True)  # the code is in the input dataset already
shutil.rmtree(WORK / "hf_data", ignore_errors=True)
sh(f"du -sh {WORK}/* || true")